# Medewerkerroutes Heerlen — 20 medewerkers, 5 cliënten per medewerker

Dit notebook berekent de optimale dagroutes voor **20 medewerkers**:
- **Start**: thuisadres medewerker
- **5 stops**: optimaal gekozen cliënten uit `clients.csv` (100 beschikbaar)
- **Einde**: terug naar huis

Aanpak:
1. Laad het wegennet en bouw een NetworkX-graph.
2. Laad medewerkers (thuis) en cliënten — koppel beiden aan het dichtstbijzijnde netwerkknooppunt.
3. Wijs 5 unieke cliënten toe aan elke medewerker (nearest-neighbor greedy, zodat elke cliënt max. 1× bezocht wordt).
4. Optimaliseer de volgorde van de 5 stops per medewerker (nearest-neighbor TSP: thuis → stop1 → … → stop5 → thuis).
5. Teken de routes op een interactieve Folium-kaart met 20 unieke kleuren.

## 1. Bibliotheken importeren

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from shapely import wkt
import folium
from scipy.spatial import cKDTree
from itertools import permutations
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## 2. Wegennet laden en graph bouwen

In [ ]:
edges_df = pd.read_csv('../output/heerlen_edge_table.csv')
print(f'Edges loaded: {len(edges_df)}')
edges_df['geometry'] = edges_df['geometry'].apply(wkt.loads)

G = nx.Graph()
node_coords = {}  # node_id -> (lon, lat)

for _, row in edges_df.iterrows():
    geom = row['geometry']
    coords = list(geom.coords)
    u, v = row['u'], row['v']
    G.add_edge(u, v, weight=row['travel_time_min'], geometry=geom)
    node_coords[u] = (coords[0][0],  coords[0][1])
    node_coords[v] = (coords[-1][0], coords[-1][1])

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.')

# k-d tree for fast nearest-node lookup
node_ids      = list(node_coords.keys())
node_lons_arr = np.array([node_coords[n][0] for n in node_ids])
node_lats_arr = np.array([node_coords[n][1] for n in node_ids])
tree = cKDTree(np.column_stack((node_lons_arr, node_lats_arr)))

# Edge geometry lookup (both directions)
edge_geom = {}
for _, row in edges_df.iterrows():
    edge_geom[(row['u'], row['v'])] = row['geometry']
    edge_geom[(row['v'], row['u'])] = row['geometry']

## 3. Medewerkers laden

In [ ]:
# ── Helper: snap a (lon, lat) pair to the nearest graph node ──────────────
def nearest_node(lon, lat):
    _, idx = tree.query([lon, lat])
    return node_ids[idx]

# ── Employee coordinates (manually geocoded from Heerlen addresses) ────────
# Source: employees.csv  (addresses are all in Heerlen, NL)
employee_data = [
    ('employees 1',  50.8872, 5.9812),
    ('employees 2',  50.8895, 5.9820),
    ('employees 3',  50.8883, 5.9830),
    ('employees 4',  50.8855, 5.9795),
    ('employees 5',  50.8945, 5.9660),
    ('employees 6',  50.8878, 5.9808),
    ('employees 7',  50.8948, 5.9700),
    ('employees 8',  50.8870, 5.9825),
    ('employees 9',  50.8868, 5.9817),
    ('employees 10', 50.8785, 5.9750),
    ('employees 11', 50.8840, 5.9810),
    ('employees 12', 50.8860, 5.9835),
    ('employees 13', 50.8850, 5.9880),
    ('employees 14', 50.8890, 5.9822),
    ('employees 15', 50.8710, 5.9920),
    ('employees 16', 50.8810, 5.9680),
    ('employees 17', 50.8952, 5.9672),
    ('employees 18', 50.8875, 5.9805),
    ('employees 19', 50.8940, 5.9665),
    ('employees 20', 50.8790, 5.9760),
]
employees_df = pd.DataFrame(employee_data, columns=['name', 'lat', 'lon'])
employees_df['home_node'] = employees_df.apply(
    lambda r: nearest_node(r['lon'], r['lat']), axis=1
)
print(f'Employees loaded: {len(employees_df)}')
print(employees_df[['name', 'lat', 'lon', 'home_node']].to_string())

## 4. Cliënten laden uit clients.csv

In [ ]:
clients_df = pd.read_csv('../output/clients.csv')
print('Original columns:', clients_df.columns.tolist())
print('First 3 rows:\n', clients_df.head(3))

# Auto-detect coordinate column (string containing two numbers)
coord_col = None
for col in clients_df.columns:
    sample = clients_df[col].dropna().astype(str).iloc[0]
    parts = sample.replace(',', ' ').replace(';', ' ').split()
    if len(parts) == 2:
        try:
            float(parts[0]); float(parts[1])
            coord_col = col
            break
        except ValueError:
            pass

if coord_col is None:
    coord_col = clients_df.columns[0]
    print(f'No coordinate column detected, using first column: {coord_col}')
else:
    print(f'Using coordinate column: "{coord_col}"')

def split_coords(s):
    parts = str(s).replace(';', ' ').replace(',', ' ').split()
    if len(parts) == 2:
        return float(parts[0]), float(parts[1])
    return np.nan, np.nan

clients_df[['lat', 'lon']] = clients_df[coord_col].apply(
    lambda x: pd.Series(split_coords(x))
)
clients_df = clients_df.dropna(subset=['lat', 'lon']).reset_index(drop=True)
clients_df['client_id'] = clients_df.index  # stable integer ID
clients_df['node'] = clients_df.apply(
    lambda r: nearest_node(r['lon'], r['lat']), axis=1
)

print(f'Clients loaded: {len(clients_df)}')
print(clients_df[['client_id', 'lat', 'lon', 'node']].head())

## 5. Reistijdenmatrix berekenen

We berekenen de kortste reistijd (Dijkstra) vanuit elk thuisknooppunt én elk cliëntknooppunt naar alle andere knooppunten.  
Dit geeft ons alle benodigde pairwise reistijden.

In [ ]:
# Collect all unique nodes we need shortest paths FROM
all_source_nodes = set(employees_df['home_node'].tolist() + clients_df['node'].tolist())
print(f'Computing Dijkstra from {len(all_source_nodes)} unique source nodes...')

# dist_lookup[src][dst] = travel time in minutes
dist_lookup = {}
for i, src in enumerate(all_source_nodes):
    dist_lookup[src] = nx.single_source_dijkstra_path_length(G, src, weight='weight')
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(all_source_nodes)} done')

def travel_time(node_a, node_b):
    """Return travel time (min) between two graph nodes."""
    return dist_lookup.get(node_a, {}).get(node_b, float('inf'))

print('Distance lookup ready.')

## 6. Cliënttoewijzing — greedy nearest-neighbor

Elke cliënt mag maximaal **één keer** bezocht worden.  
We lopen per medewerker in volgorde en kennen steeds de 5 dichtstbijzijnde nog-beschikbare cliënten toe.

In [ ]:
N_STOPS = 5  # clients per employee

available = set(clients_df['client_id'].tolist())  # pool of unassigned clients
assignments = {}  # emp_idx -> list of client_ids

for emp_idx, emp in employees_df.iterrows():
    home_node = emp['home_node']
    # Score all available clients by travel time from home
    scored = [
        (travel_time(home_node, clients_df.loc[cid, 'node']), cid)
        for cid in available
    ]
    scored.sort()
    chosen = [cid for _, cid in scored[:N_STOPS]]
    assignments[emp_idx] = chosen
    available -= set(chosen)
    print(f'{emp["name"]}: clients {chosen}')

print(f'\nClients remaining unassigned: {len(available)}')

## 7. Routevolgorde optimaliseren per medewerker (nearest-neighbor TSP)

Voor elke medewerker optimaliseren we de volgorde van de 5 stops met een **nearest-neighbor heuristiek**:  
Start thuis → bezoek steeds de dichtstbijzijnde nog-niet-bezochte stop → keer terug naar huis.

In [ ]:
def nn_tour(home_node, client_nodes):
    """
    Nearest-neighbor TSP heuristic.
    Returns ordered list of nodes: [home, stop1, stop2, ..., stopN, home]
    and the total travel time.
    """
    unvisited = list(client_nodes)
    tour = [home_node]
    current = home_node
    total_time = 0.0

    while unvisited:
        # Find nearest unvisited node
        best_time = float('inf')
        best_node = None
        for node in unvisited:
            t = travel_time(current, node)
            if t < best_time:
                best_time = t
                best_node = node
        tour.append(best_node)
        total_time += best_time
        unvisited.remove(best_node)
        current = best_node

    # Return home
    total_time += travel_time(current, home_node)
    tour.append(home_node)
    return tour, total_time


# Build tour for each employee
tours = {}  # emp_idx -> {'tour_nodes': [...], 'total_time': float, 'client_nodes': [...]}

for emp_idx, emp in employees_df.iterrows():
    home_node = emp['home_node']
    chosen_ids = assignments[emp_idx]
    chosen_nodes = [clients_df.loc[cid, 'node'] for cid in chosen_ids]

    tour_nodes, total_time = nn_tour(home_node, chosen_nodes)
    tours[emp_idx] = {
        'tour_nodes':   tour_nodes,
        'client_nodes': chosen_nodes,
        'client_ids':   chosen_ids,
        'total_time':   total_time,
    }
    print(f'{emp["name"]}: {total_time:.1f} min total, stops={len(chosen_nodes)}')

## 8. Kaart bouwen

Elke medewerker krijgt een unieke kleur. De route volgt de echte wegen uit het wegennet.

In [ ]:
# 20 visually distinct colors
EMPLOYEE_COLORS = [
    '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45', '#fabed4',
    '#469990', '#dcbeff', '#9A6324', '#ff8c00', '#800000',
    '#aaffc3', '#808000', '#00bfff', '#000075', '#a9a9a9',
]

def road_path_latlon(node_sequence):
    """Convert a node sequence to a list of (lat, lon) coordinates along actual roads."""
    result = []
    for i in range(len(node_sequence) - 1):
        u, v = node_sequence[i], node_sequence[i + 1]
        geom = edge_geom.get((u, v))
        if geom is not None:
            result.extend([(lat, lon) for lon, lat in geom.coords])
        else:
            # Straight-line fallback
            cu, cv = node_coords.get(u), node_coords.get(v)
            if cu and cv:
                result += [(cu[1], cu[0]), (cv[1], cv[0])]
    return result

def get_segment_path(node_a, node_b):
    """Get the road-following path between two graph nodes (full shortest path)."""
    try:
        path = nx.shortest_path(G, source=node_a, target=node_b, weight='weight')
        return road_path_latlon(path)
    except nx.NetworkXNoPath:
        # Straight line fallback
        ca, cb = node_coords.get(node_a), node_coords.get(node_b)
        if ca and cb:
            return [(ca[1], ca[0]), (cb[1], cb[0])]
        return []

# Map center
center_lat = np.mean(node_lats_arr)
center_lon = np.mean(node_lons_arr)

m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB positron')

# ── Road network background ──────────────────────────────────────────────
for _, row in edges_df.iterrows():
    latlon = [(lat, lon) for lon, lat in row['geometry'].coords]
    folium.PolyLine(locations=latlon, color='#cccccc', weight=1, opacity=0.35).add_to(m)

# ── Employee routes ──────────────────────────────────────────────────────
for emp_idx, emp in employees_df.iterrows():
    color   = EMPLOYEE_COLORS[emp_idx % len(EMPLOYEE_COLORS)]
    tour    = tours[emp_idx]
    t_nodes = tour['tour_nodes']  # [home, s1, s2, s3, s4, s5, home]

    for seg_i in range(len(t_nodes) - 1):
        latlon = get_segment_path(t_nodes[seg_i], t_nodes[seg_i + 1])
        if latlon:
            label = 'thuis → stop' if seg_i == 0 else (
                f'stop{seg_i} → stop{seg_i+1}' if seg_i < len(t_nodes) - 2
                else f'stop{seg_i} → thuis'
            )
            folium.PolyLine(
                locations=latlon,
                color=color,
                weight=4,
                opacity=0.82,
                tooltip=f"{emp['name']} | {label}"
            ).add_to(m)

# ── Home markers ────────────────────────────────────────────────────────
for emp_idx, emp in employees_df.iterrows():
    color = EMPLOYEE_COLORS[emp_idx % len(EMPLOYEE_COLORS)]
    tour  = tours[emp_idx]
    folium.Marker(
        location=[emp['lat'], emp['lon']],
        icon=folium.DivIcon(
            html=f'<div style="width:20px;height:20px;background:{color};'
                 'border:2px solid white;border-radius:50%;'
                 'box-shadow:0 1px 4px rgba(0,0,0,.4);"></div>',
            icon_size=(20, 20),
            icon_anchor=(10, 10)
        ),
        popup=folium.Popup(
            f"<b>{emp['name']}</b><br>"
            f"Totale reistijd: {tour['total_time']:.1f} min<br>"
            f"Cliënten: {tour['client_ids']}",
            max_width=220
        ),
        tooltip=f"{emp['name']} (thuis)"
    ).add_to(m)

# ── Client markers ───────────────────────────────────────────────────────
# Map client_id -> employee color
client_color_map = {}
for emp_idx in range(len(employees_df)):
    color = EMPLOYEE_COLORS[emp_idx % len(EMPLOYEE_COLORS)]
    for cid in tours[emp_idx]['client_ids']:
        client_color_map[cid] = (color, employees_df.loc[emp_idx, 'name'])

for _, client in clients_df.iterrows():
    cid   = client['client_id']
    color, emp_name = client_color_map.get(cid, ('#888888', 'Niet toegewezen'))
    stop_num = tours[next(
        (ei for ei in range(len(employees_df)) if cid in tours[ei]['client_ids']), 0
    )]['client_ids'].index(cid) + 1 if cid in client_color_map else '–'

    folium.CircleMarker(
        location=[client['lat'], client['lon']],
        radius=6,
        color='white',
        weight=1.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(
            f"<b>Cliënt {cid}</b><br>"
            f"Medewerker: {emp_name}<br>"
            f"Stop #{stop_num}",
            max_width=180
        ),
        tooltip=f"Cliënt {cid} | {emp_name}"
    ).add_to(m)

print('Map built successfully.')

## 9. Legenda toevoegen en kaart opslaan

In [ ]:
legend_rows = ''
for emp_idx, emp in employees_df.iterrows():
    color = EMPLOYEE_COLORS[emp_idx % len(EMPLOYEE_COLORS)]
    t     = tours[emp_idx]['total_time']
    cids  = tours[emp_idx]['client_ids']
    legend_rows += (
        '<tr>'
        f'<td><div style="width:14px;height:14px;background:{color};'
        'border:1px solid #ccc;border-radius:3px;"></div></td>'
        f'<td style="padding:0 6px;white-space:nowrap;">{emp["name"]}</td>'
        f'<td style="color:#555;white-space:nowrap;">{t:.0f} min | '
        f'cliënten: {cids}</td>'
        '</tr>'
    )

legend_html = (
    '<div style="position:fixed;bottom:20px;left:20px;z-index:1000;'
    'background:white;padding:10px 14px;border-radius:7px;'
    'font-size:11px;font-family:sans-serif;'
    'box-shadow:0 2px 10px rgba(0,0,0,.3);'
    'max-height:460px;overflow-y:auto;">'
    '<b style="font-size:13px;">Routeoverzicht</b>'
    '<table style="border-collapse:collapse;margin-top:6px;line-height:1.6;">'
    '<tr><th style="text-align:left;padding-right:6px;">Kleur</th>'
    '<th style="text-align:left;">Medewerker</th>'
    '<th style="text-align:left;padding-left:6px;">Totale tijd &amp; stops</th></tr>'
    + legend_rows +
    '</table>'
    '<hr style="margin:8px 0;">'  
    '<span style="color:#555;">● = cliënt &nbsp;&nbsp; &#9679; = thuisadres (medewerker)</span>'
    '</div>'
)

m.get_root().html.add_child(folium.Element(legend_html))

output_path = '../output/employee_routes_map.html'
m.save(output_path)
print(f'Map saved: {output_path}')

from IPython.display import IFrame, display
display(IFrame(src='employee_routes_map.html', width='100%', height=680))

## 10. Samenvatting per medewerker

In [ ]:
print('=== Routesamenvatting ===')
rows = []
for emp_idx, emp in employees_df.iterrows():
    t = tours[emp_idx]
    rows.append({
        'Medewerker':     emp['name'],
        'Cliënt-IDs':     str(t['client_ids']),
        'Totale tijd (min)': round(t['total_time'], 1),
    })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(f'\nGemiddelde totale reistijd: {summary["Totale tijd (min)"].mean():.1f} min')